# ForestGuard

## 03 — Análisis exploratorio

Análisis del dataset integrado y procesado de incendios forestales 2022–2025.

## 1. Carga del dataset procesado

In [ ]:
from pathlib import Path
from google.colab import files
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/content")
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

archivo_processed = PROCESSED / "incendios_forestales_2022_2025_limpio.csv"

if not archivo_processed.exists():
    print("Sube el archivo generado por el módulo 02:")
    print("- incendios_forestales_2022_2025_limpio.csv")

    subidos = files.upload()

    for nombre, contenido in subidos.items():
        with open(PROCESSED / nombre, "wb") as f:
            f.write(contenido)

if not archivo_processed.exists():
    raise FileNotFoundError(
        "No se encontró el dataset procesado. "
        "Ejecuta el módulo 02 o sube el CSV generado."
    )

df = pd.read_csv(archivo_processed)
print("Dataset cargado:", df.shape)

## 2. Validación inicial

In [ ]:
validacion = pd.DataFrame({
    "Indicador": [
        "Registros",
        "Columnas",
        "Duplicados",
        "Valores faltantes",
        "Año mínimo",
        "Año máximo",
    ],
    "Resultado": [
        df.shape[0],
        df.shape[1],
        df.duplicated().sum(),
        int(df.isnull().sum().sum()),
        df["año"].min(),
        df["año"].max(),
    ],
})

display(validacion)

## 3. Cantidad de registros por año

Esta visualización permite comprobar el aumento de datos y conocer cuánto aporta cada año al dataset integrado.

In [ ]:
registros_anio = (
    df["año"]
    .value_counts()
    .sort_index()
)

display(
    registros_anio.to_frame(
        "Cantidad de incendios"
    )
)

plt.figure(figsize=(8, 5))
registros_anio.plot(kind="bar")
plt.title("Incendios forestales registrados por año")
plt.xlabel("Año")
plt.ylabel("Número de registros")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Incendios por provincia

In [ ]:
conteo_provincia = (
    df["provincia"]
    .value_counts()
    .head(15)
    .sort_values()
)

display(
    conteo_provincia.to_frame(
        "Cantidad de incendios"
    )
)

plt.figure(figsize=(9, 6))
conteo_provincia.plot(kind="barh")
plt.title("Incendios forestales registrados por provincia")
plt.xlabel("Número de registros")
plt.ylabel("Provincia")
plt.tight_layout()
plt.show()

## 5. Distribución por región

In [ ]:
conteo_region = (
    df["region"]
    .value_counts(dropna=False)
    .sort_values()
)

display(
    conteo_region.to_frame(
        "Cantidad de incendios"
    )
)

plt.figure(figsize=(8, 5))
conteo_region.plot(kind="barh")
plt.title("Incendios forestales registrados por región")
plt.xlabel("Número de registros")
plt.ylabel("Región")
plt.tight_layout()
plt.show()

## 6. Distribución mensual por año

Se agrupan los incendios por mes y año para observar si el comportamiento cambia entre los diferentes periodos.

In [ ]:
incendios_mes_anio = (
    df.groupby(["mes", "año"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

display(incendios_mes_anio)

plt.figure(figsize=(10, 5))

for anio in incendios_mes_anio.columns:
    plt.plot(
        incendios_mes_anio.index,
        incendios_mes_anio[anio],
        marker="o",
        label=str(anio)
    )

plt.title("Incendios forestales por mes y año")
plt.xlabel("Mes")
plt.ylabel("Número de registros")
plt.xticks(range(1, 13))
plt.legend(title="Año")
plt.tight_layout()
plt.show()

## 7. Hectáreas afectadas por año

In [ ]:
hectareas_anio = (
    df.groupby("año")["hadecobert"]
    .agg(["count", "mean", "median", "sum", "max"])
    .round(2)
)

display(hectareas_anio)

total_hectareas = (
    df.groupby("año")["hadecobert"]
    .sum(min_count=1)
)

plt.figure(figsize=(8, 5))
total_hectareas.plot(kind="bar")
plt.title("Hectáreas afectadas registradas por año")
plt.xlabel("Año")
plt.ylabel("Hectáreas")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Nivel de afectación

Se revisa la distribución del nivel de afectación disponible en las fuentes. Los valores faltantes se conservan porque no todos los años necesariamente registran esta variable con el mismo nivel de detalle.

In [ ]:
nivel_anio = pd.crosstab(
    df["año"],
    df["nivelinf"],
    dropna=False
)

display(nivel_anio)

## 9. Variables adicionales de impacto

Con la integración se incorporan variables que permiten analizar no solo el área quemada, sino también posibles afectaciones a personas, familias y animales.

In [ ]:
variables_impacto = [
    "personas_afectadas",
    "familias_afectadas",
    "animales_afectados",
    "animales_muertos",
]

disponibles = [
    c for c in variables_impacto
    if c in df.columns
]

impacto_anio = (
    df.groupby("año")[disponibles]
    .sum(min_count=1)
    .round(2)
)

display(impacto_anio)

## 10. Principales causas registradas

In [ ]:
causas = (
    df["causa"]
    .fillna("Sin información")
    .value_counts()
    .head(10)
    .sort_values()
)

display(
    causas.to_frame(
        "Cantidad de registros"
    )
)

plt.figure(figsize=(9, 6))
causas.plot(kind="barh")
plt.title("Principales causas registradas")
plt.xlabel("Número de registros")
plt.ylabel("Causa")
plt.tight_layout()
plt.show()

## 11. Matriz de correlación

La correlación se utiliza únicamente como una referencia exploratoria. No implica causalidad.

In [ ]:
variables_correlacion = [
    "año",
    "mes",
    "dia",
    "latitud",
    "longitud",
    "hadecobert",
    "personas_afectadas",
    "familias_afectadas",
    "animales_afectados",
    "animales_muertos",
]

variables_correlacion = [
    c for c in variables_correlacion
    if c in df.columns
]

correlacion = (
    df[variables_correlacion]
    .corr(numeric_only=True)
)

display(correlacion)

plt.figure(figsize=(10, 8))
plt.imshow(correlacion, aspect="auto")
plt.xticks(
    range(len(correlacion.columns)),
    correlacion.columns,
    rotation=90
)
plt.yticks(
    range(len(correlacion.columns)),
    correlacion.columns
)
plt.colorbar()
plt.title("Matriz de correlación")
plt.tight_layout()
plt.show()

## 12. Flujo del Pipeline de Datos

El pipeline actualizado integra dos fuentes oficiales y cuatro años de información.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3))
ax.axis("off")

etapas = [
    ("API histórica\n2022–2024", 0.07),
    ("API incendios\n2025", 0.24),
    ("Extracción", 0.40),
    ("Homologación\ne integración", 0.57),
    ("Limpieza y\ntransformación", 0.74),
    ("EDA / dataset\npara ML", 0.91),
]

for texto, x in etapas:
    ax.text(
        x,
        0.5,
        texto,
        ha="center",
        va="center",
        bbox=dict(
            boxstyle="round,pad=0.5",
            fill=False
        ),
        transform=ax.transAxes,
    )

for i in range(len(etapas) - 1):
    ax.annotate(
        "",
        xy=(etapas[i + 1][1] - 0.07, 0.5),
        xytext=(etapas[i][1] + 0.07, 0.5),
        xycoords=ax.transAxes,
        arrowprops=dict(arrowstyle="->"),
    )

plt.title("Pipeline de Datos actualizado de ForestGuard")
plt.show()

## 13. Conclusión

La actualización permite trabajar con registros correspondientes a 2022, 2023, 2024 y 2025, en lugar de limitar el análisis a un solo año.

También se incorporan variables adicionales de impacto relacionadas con personas, familias y animales, además de mantener variables geográficas, temporales, hectáreas afectadas y nivel de afectación.

Esta ampliación responde a la necesidad de contar con una base de datos más completa para los análisis posteriores de ForestGuard.